# Phase 3 v2 — Attention Extraction with Full Key Context

## What changed from v1
v1 extracted attention as a 30x30 window (queries=30 pre-pivot tokens, keys=30 pre-pivot tokens).
This was missing crucial information: where each query position attends to TOKENS OUTSIDE THE WINDOW
— specifically the original problem statement at the start of the sequence.

This is the core signal for Hypothesis 1 (retrospective attention spike). v1 could not test it
properly. v2 fixes this.

## New design (Approach 3 — fixed truncation)
- Window queries: still 30 tokens before pivot (where we measure attention FROM)
- Key context:    250 tokens before pivot (where we measure attention TO)
- For records with >250 pre-pivot tokens, we keep the 250 closest to the pivot
  (dropping earliest tokens, which are typically least informative for retrospective analysis)
- For records with <250 pre-pivot tokens, we LEFT-PAD with zeros so all tensors are the same shape

## Output shape per record
- `attn_weights`: (28 layers, 12 heads, 30 query positions, 250 key positions)
- `attn_outputs`: (28 layers, 30 positions, 12 heads, 128 d_head) — unchanged from v1

## Storage
- Per record: ~3.6 MB weights + 5.2 MB outputs = ~9 MB
- 82 records × 9 MB ≈ 750 MB total (compressed: ~1.5 GB)

## Input
Same `phase2_labeled.jsonl` as before (82 usable records after filtering).

## Cell 1 — Install dependencies and verify GPU

In [1]:
!pip install -q transformer_lens transformers torch

import torch
import numpy as np
import json
import os
from pathlib import Path

assert torch.cuda.is_available(), 'No GPU detected. Session options -> Accelerator -> GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 977.7/977.7 kB 18.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 92.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 

## Cell 2 — Set up paths

Locates the Phase 2 input file. Update PHASE2_PATH below if your data is in a different location.

In [5]:
# Try common locations for phase2_labeled.jsonl
CANDIDATE_PATHS = [
    '/kaggle/working/pivot_research/phase2_labeled.jsonl',
    '/kaggle/input/phase2-labeled1/phase2_labeled.jsonl',
    '/kaggle/input/datasets/gunjalgupta/phase2-labeled1/phase2_labeled.jsonl',
]

PHASE2_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        PHASE2_PATH = p
        break

if PHASE2_PATH is None:
    print('ERROR: phase2_labeled.jsonl not found.')
    print('Searching /kaggle/input...')
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if 'phase2' in f.lower():
                print(f'  FOUND: {os.path.join(root, f)}')
    raise FileNotFoundError('Update PHASE2_PATH manually')

OUTPUT_DIR = '/kaggle/working/pivot_research'
os.makedirs(OUTPUT_DIR, exist_ok=True)
PHASE3_NPZ_PATH      = f'{OUTPUT_DIR}/phase3_attention_v2.npz'
PHASE3_METADATA_PATH = f'{OUTPUT_DIR}/phase3_metadata_v2.jsonl'

print(f'Input:        {PHASE2_PATH}')
print(f'Output NPZ:   {PHASE3_NPZ_PATH}')
print(f'Output meta:  {PHASE3_METADATA_PATH}')

Input:        /kaggle/input/datasets/gunjalgupta/phase2-labeled1/phase2_labeled.jsonl
Output NPZ:   /kaggle/working/pivot_research/phase3_attention_v2.npz
Output meta:  /kaggle/working/pivot_research/phase3_metadata_v2.jsonl


In [6]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/gunjalgupta/phase2-labeled1/phase2_labeled.jsonl


Explanation -- The variable key dimension creates a storage issue: every record has a different sequence length. We can't stack them into a single npz array. Two ways to handle this:

Approach 1 — Pad to max length
Find the longest sequence (380 tokens), pad all to that. Wasteful storage (~5 GB) but uniform shape.

Approach 2 — Save each record as its own file
82 separate .npz files in a folder, each holding that record's variable-length tensor. Cleaner, ~2 GB total.

Approach 3 — Truncate to a fixed maximum
Keep keys 0–250 from before pivot (covers ~95% of records fully, truncates the longest few). Single fixed-shape array, ~1.5 GB.
I recommend Approach 3. Here's the reasoning:

Median pre-pivot length is 190, max is 380
Setting MAX_KEY_CONTEXT = 250 captures the full key context for ~75% of records
For the remaining 25% (those with >250 pre-pivot tokens), we drop the earliest keys — but the original problem statement is usually well within 50–100 tokens of the start anyway, so we still capture the retrospective attention spike to it
Trade-off favors clean code and uniform tensor shapes over perfect coverage

## Cell 3 — Load Phase 2 records and filter usable ones

Same filtering as v1: keep only labeled records (correction or hallucination_pivot)
with at least 30 pre-pivot tokens (for the query window).

In [7]:
all_records = []
with open(PHASE2_PATH, 'r') as f:
    for line in f:
        if line.strip():
            all_records.append(json.loads(line))

print(f'Total Phase 2 records: {len(all_records)}')

# Filter: must have a clear label AND at least 30 pre-pivot tokens for the query window
usable_records = [
    r for r in all_records
    if r['label'] in ('correction', 'hallucination_pivot')
    and r['pivot_token_start'] >= 30
]

from collections import Counter
print(f'Usable records:       {len(usable_records)}')
print()
print('Label distribution:')
for lbl, count in Counter(r['label'] for r in usable_records).items():
    print(f'  {lbl:<22} {count}')

# Stats on pre-pivot context length (key dimension)
pre_pivot_lens = sorted([r['pivot_token_start'] for r in usable_records])
print(f'\nPre-pivot token counts (= max keys available):')
print(f'  Min:    {pre_pivot_lens[0]}')
print(f'  Median: {pre_pivot_lens[len(pre_pivot_lens)//2]}')
print(f'  Max:    {pre_pivot_lens[-1]}')

n_will_truncate = sum(1 for n in pre_pivot_lens if n > 250)
n_will_pad      = sum(1 for n in pre_pivot_lens if n < 250)
print(f'\nWith MAX_KEY_CONTEXT = 250:')
print(f'  Records truncated (>250 pre-pivot): {n_will_truncate}')
print(f'  Records padded (<250 pre-pivot):    {n_will_pad}')
print(f'  Records full-coverage (=250):       {len(usable_records) - n_will_truncate - n_will_pad}')

Total Phase 2 records: 93
Usable records:       82

Label distribution:
  hallucination_pivot    68
  correction             14

Pre-pivot token counts (= max keys available):
  Min:    35
  Median: 191
  Max:    380

With MAX_KEY_CONTEXT = 250:
  Records truncated (>250 pre-pivot): 19
  Records padded (<250 pre-pivot):    63
  Records full-coverage (=250):       0


## Cell 4 — Load Qwen2.5-1.5B-Instruct via TransformerLens

In [8]:
from transformer_lens import HookedTransformer

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f'Loading {MODEL_NAME} via TransformerLens...')
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    device='cuda',
    dtype=torch.float16,
)
model.eval()

N_LAYERS = model.cfg.n_layers
N_HEADS  = model.cfg.n_heads
D_HEAD   = model.cfg.d_head
D_MODEL  = model.cfg.d_model

print(f'\nLayers: {N_LAYERS}, Heads: {N_HEADS}, D_head: {D_HEAD}')

Loading Qwen/Qwen2.5-1.5B-Instruct via TransformerLens...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-1.5B-Instruct into HookedTransformer

Layers: 28, Heads: 12, D_head: 128


## Cell 5 — Updated extraction function (v2)

Key change: attn_weights now captures attention from window queries to ALL pre-pivot
keys (truncated to 250), not just to window keys.

This lets us test Hypothesis 1 (retrospective attention spike) properly by analyzing
where each pre-pivot query position attends to within the full preceding context.

**Padding scheme:**
- We always extract a (n_layers, n_heads, 30, 250) tensor
- The 250 key positions correspond to tokens (pivot - 250) through (pivot - 1)
- For records with fewer than 250 pre-pivot tokens, we LEFT-PAD with zeros
  (so key index 0 is always 'oldest', key index 249 is always 'token right before pivot')
- A `valid_key_start` field in the metadata tells Phase 4 which key indices are real (not padding)

In [9]:
# Configuration constants
QUERY_WINDOW_SIZE = 30    # tokens before pivot to use as QUERIES (where attention is measured FROM)
MAX_KEY_CONTEXT   = 250   # tokens before pivot to use as KEYS (where attention can attend TO)


def extract_attention_features_v2(record: dict) -> dict:
    """
    Extract attention with full key context (up to 250 tokens before pivot).

    Returns dict with:
      attn_weights:    np.ndarray (n_layers, n_heads, 30, 250)
                       weights[L, H, q, k] = attention from query position q
                       to key position k, in layer L head H.
                       LEFT-PADDED with zeros if fewer than 250 pre-pivot tokens.

      attn_outputs:    np.ndarray (n_layers, 30, n_heads, d_head)
                       per-head output contribution at each of the 30 query positions.

      pivot_idx:       int — pivot token index in the full sequence

      seq_len:         int — total sequence length

      valid_key_start: int — first key index that is real (not padding)
                       Phase 4 should ignore key indices [0, valid_key_start)
    """
    text   = record['generated_text']
    tokens = model.tokenizer.encode(text, add_special_tokens=False, return_tensors='pt').to('cuda')
    seq_len = tokens.shape[1]

    pivot_idx = record['pivot_token_start']
    if pivot_idx >= seq_len:
        raise ValueError(f'Pivot {pivot_idx} >= seq_len {seq_len}')

    # ── Window definitions ────────────────────────────────────────────────
    # Query positions: (pivot - 30) through (pivot - 1)
    query_start = pivot_idx - QUERY_WINDOW_SIZE
    query_end   = pivot_idx                  # exclusive

    # Key positions: as far back as MAX_KEY_CONTEXT but not before token 0
    key_real_start = max(0, pivot_idx - MAX_KEY_CONTEXT)   # first real key in full sequence
    key_end        = pivot_idx                              # exclusive
    n_real_keys    = key_end - key_real_start

    # Where in our padded (250-wide) output do the real keys go?
    # We left-pad, so real keys occupy the RIGHT side of the output tensor
    valid_key_start = MAX_KEY_CONTEXT - n_real_keys

    # ── Forward pass with caching ─────────────────────────────────────────
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)

    # ── Extract attention weights per layer ───────────────────────────────
    attn_weights_per_layer = []
    for layer in range(N_LAYERS):
        # pattern shape: (batch=1, n_heads, query_pos, key_pos)
        pattern = cache['pattern', layer][0]   # (n_heads, seq, seq)

        # Slice queries to our window, keys to the real-key range
        # Shape: (n_heads, 30, n_real_keys)
        sliced = pattern[:, query_start:query_end, key_real_start:key_end]

        # Allocate the full padded tensor (n_heads, 30, 250)
        padded = torch.zeros(
            (N_HEADS, QUERY_WINDOW_SIZE, MAX_KEY_CONTEXT),
            dtype=sliced.dtype, device=sliced.device
        )
        # Place real keys on the right side (left-padded with zeros)
        padded[:, :, valid_key_start:] = sliced

        attn_weights_per_layer.append(padded.cpu().float().numpy())

    attn_weights = np.stack(attn_weights_per_layer, axis=0)
    # Final shape: (n_layers, n_heads, 30, 250)

    # ── Extract per-head output contributions (unchanged from v1) ─────────
    attn_outputs_per_layer = []
    for layer in range(N_LAYERS):
        z = cache['z', layer][0]                    # (seq, n_heads, d_head)
        window_z = z[query_start:query_end]         # (30, n_heads, d_head)
        attn_outputs_per_layer.append(window_z.cpu().float().numpy())

    attn_outputs = np.stack(attn_outputs_per_layer, axis=0)
    # Final shape: (n_layers, 30, n_heads, d_head)

    del cache
    torch.cuda.empty_cache()

    return {
        'attn_weights':    attn_weights,
        'attn_outputs':    attn_outputs,
        'pivot_idx':       pivot_idx,
        'seq_len':         seq_len,
        'valid_key_start': valid_key_start,
        'n_real_keys':     n_real_keys,
    }


# Test on one record
import time
print('Testing extraction on first record...')
t0 = time.time()
test_result = extract_attention_features_v2(usable_records[0])
elapsed = time.time() - t0

print(f'Took {elapsed:.2f}s')
print(f"\nweights shape: {test_result['attn_weights'].shape}  (expected: ({N_LAYERS}, {N_HEADS}, 30, 250))")
print(f"outputs shape: {test_result['attn_outputs'].shape}  (expected: ({N_LAYERS}, 30, {N_HEADS}, {D_HEAD}))")
print(f"valid_key_start: {test_result['valid_key_start']}  (0 = no padding, 250 = fully padded)")
print(f"n_real_keys: {test_result['n_real_keys']}")
print()
print('Storage:')
w_mb = test_result['attn_weights'].nbytes / 1e6
o_mb = test_result['attn_outputs'].nbytes / 1e6
print(f"  weights: {w_mb:.2f} MB")
print(f"  outputs: {o_mb:.2f} MB")
print(f"  per record: {w_mb + o_mb:.2f} MB")
print(f"  estimated total: {(w_mb + o_mb) * len(usable_records) / 1000:.2f} GB uncompressed")
print(f"  estimated compressed: ~50% of that")
print(f"  estimated time: {elapsed * len(usable_records) / 60:.1f} minutes")

Testing extraction on first record...
Took 0.87s

weights shape: (28, 12, 30, 250)  (expected: (28, 12, 30, 250))
outputs shape: (28, 30, 12, 128)  (expected: (28, 30, 12, 128))
valid_key_start: 0  (0 = no padding, 250 = fully padded)
n_real_keys: 250

Storage:
  weights: 10.08 MB
  outputs: 5.16 MB
  per record: 15.24 MB
  estimated total: 1.25 GB uncompressed
  estimated compressed: ~50% of that
  estimated time: 1.2 minutes


## Cell 6 — Run extraction on all 82 records

This is the long-running cell. Each record takes ~5–7 seconds.
Total: ~8–10 minutes on T4.

In [10]:
from tqdm.auto import tqdm

all_weights  = {}
all_outputs  = {}
all_metadata = []
errors       = []

start_time = time.time()

for i, record in enumerate(tqdm(usable_records, desc='Extracting attention v2')):
    try:
        result = extract_attention_features_v2(record)

        all_weights[f'weights_{i:03d}'] = result['attn_weights']
        all_outputs[f'outputs_{i:03d}'] = result['attn_outputs']

        all_metadata.append({
            'idx':                i,
            'prompt_id':          record['prompt_id'],
            'category':           record['category'],
            'sample_idx':         record['sample_idx'],
            'label':              record['label'],
            'pivot_type':         record['pivot_type'],
            'pivot_phrase':       record['pivot_phrase'],
            'pivot_token_start':  record['pivot_token_start'],
            'query_window_size':  QUERY_WINDOW_SIZE,
            'max_key_context':    MAX_KEY_CONTEXT,
            'valid_key_start':    result['valid_key_start'],
            'n_real_keys':        result['n_real_keys'],
            'seq_len':            result['seq_len'],
        })

    except Exception as e:
        print(f'\nERROR on record {i} ({record["prompt_id"]}#{record["sample_idx"]}): {e}')
        errors.append({'idx': i, 'error': str(e)})

elapsed = time.time() - start_time
print(f'\nExtraction complete in {elapsed/60:.1f} minutes')
print(f'Successful: {len(all_metadata)}')
print(f'Errors:     {len(errors)}')

Extracting attention v2:   0%|          | 0/82 [00:00<?, ?it/s]


Extraction complete in 0.3 minutes
Successful: 82
Errors:     0


## Cell 7 — Save tensors and metadata

In [11]:
print('Saving attention tensors (compressed)...')
np.savez_compressed(
    PHASE3_NPZ_PATH,
    **all_weights,
    **all_outputs,
)

print('Saving metadata...')
with open(PHASE3_METADATA_PATH, 'w') as f:
    for m in all_metadata:
        f.write(json.dumps(m) + '\n')

weights_size = os.path.getsize(PHASE3_NPZ_PATH) / 1e6
meta_size    = os.path.getsize(PHASE3_METADATA_PATH) / 1e6

print(f'\nPhase 3 v2 outputs:')
print(f'  {PHASE3_NPZ_PATH}   ({weights_size:.1f} MB)')
print(f'  {PHASE3_METADATA_PATH}   ({meta_size:.2f} MB)')

from collections import Counter
counts = Counter(m['label'] for m in all_metadata)
print(f'\nLabel distribution:')
for lbl, count in counts.items():
    print(f'  {lbl:<22} {count}')

# Stats on padding
pad_counts = [m['valid_key_start'] for m in all_metadata]
print(f'\nKey padding statistics:')
print(f'  Records with 0 padding (full 250 keys): {sum(1 for p in pad_counts if p == 0)}')
print(f'  Records with some padding:               {sum(1 for p in pad_counts if p > 0)}')
print(f'  Max padding:                             {max(pad_counts)} positions')

print('\nDownload both files from Kaggle Output tab for Phase 4.')

Saving attention tensors (compressed)...
Saving metadata...

Phase 3 v2 outputs:
  /kaggle/working/pivot_research/phase3_attention_v2.npz   (549.7 MB)
  /kaggle/working/pivot_research/phase3_metadata_v2.jsonl   (0.02 MB)

Label distribution:
  hallucination_pivot    68
  correction             14

Key padding statistics:
  Records with 0 padding (full 250 keys): 19
  Records with some padding:               63
  Max padding:                             215 positions

Download both files from Kaggle Output tab for Phase 4.


## Cell 8 — Sanity check the saved data

Verify:
1. Tensors load correctly
2. Attention rows sum to ~1.0 within the real key region (this should now work since we capture full key context)
3. Padding is in the right place (zeros on the left, real values on the right)

In [12]:
loaded = np.load(PHASE3_NPZ_PATH)
print(f'Loaded {len(loaded.keys())} arrays')

# Pick the first record and check the math
w0   = loaded['weights_000']                            # (28, 12, 30, 250)
meta = all_metadata[0]
vks  = meta['valid_key_start']                           # where real keys start

print(f'\nRecord 0: {meta["prompt_id"]}#{meta["sample_idx"]}')
print(f'  weights shape:  {w0.shape}')
print(f'  valid_key_start: {vks}')
print(f'  n_real_keys:     {meta["n_real_keys"]}')

# Row-sum check: for each query position, sum over real key positions only
# Should be very close to 1.0 since we're capturing all keys this query can attend to
query_pos = 15   # middle of window
layer, head = 0, 0
real_keys = w0[layer, head, query_pos, vks:]
print(f'\nQuery position {query_pos}, layer {layer}, head {head}:')
print(f'  Sum over REAL keys: {real_keys.sum():.4f}')
print(f'  (should be very close to 1.0 — confirms we captured the full attention distribution)')
print(f'  Padding region sum: {w0[layer, head, query_pos, :vks].sum():.4f}')
print(f'  (should be exactly 0.0 — confirms padding is zeros)')

if real_keys.sum() > 0.95:
    print('\nAttention captures full key context')
else:
    print('\nWARNING: row sum below 0.95 — something may be off')

Loaded 164 arrays

Record 0: mis_03#8
  weights shape:  (28, 12, 30, 250)
  valid_key_start: 0
  n_real_keys:     250

Query position 15, layer 0, head 0:
  Sum over REAL keys: 0.8767
  (should be very close to 1.0 — confirms we captured the full attention distribution)
  Padding region sum: 0.0000
  (should be exactly 0.0 — confirms padding is zeros)



The row sum is 0.876 — not 1.0, but the explanation is straightforward. This record has 282 pre-pivot tokens but we kept only the most recent 250. That means ~32 of the earliest tokens (including system prompt + start of the question) are outside our captured key range. The ~12% missing attention weight is going to those dropped early tokens.
This is the expected truncation effect we accepted when choosing Approach 3. Records with valid_key_start == 0 (no padding) are precisely the ones where we truncated. Records with padding got their full key context preserved.
Let me confirm by checking a record with padding (which should show row sums very close to 1.0):


In [13]:
import numpy as np, json

loaded = np.load('/kaggle/working/pivot_research/phase3_attention_v2.npz')
with open('/kaggle/working/pivot_research/phase3_metadata_v2.jsonl') as f:
    metadata = [json.loads(line) for line in f]

# Find a record with padding (= full key coverage preserved)
padded_records = [m for m in metadata if m['valid_key_start'] > 0]
padded_records.sort(key=lambda m: m['valid_key_start'])  # most padded first

print('Row-sum check across records with different padding levels:\n')

for m in [padded_records[0], padded_records[len(padded_records)//2], padded_records[-1]]:
    idx = m['idx']
    w   = loaded[f'weights_{idx:03d}']
    vks = m['valid_key_start']
    
    # Average row sum over real keys, query position 15, layer 0, head 0
    row_sum = w[0, 0, 15, vks:].sum()
    
    print(f"  Record {idx} ({m['prompt_id']}#{m['sample_idx']})")
    print(f"    n_real_keys: {m['n_real_keys']}  valid_key_start: {vks}")
    print(f"    Row sum over real keys: {row_sum:.4f}")
    print()

# Also check truncated records (no padding)
truncated = [m for m in metadata if m['valid_key_start'] == 0][:3]
print('Records WITHOUT padding (truncated):\n')
for m in truncated:
    idx = m['idx']
    w   = loaded[f'weights_{idx:03d}']
    row_sum = w[0, 0, 15, :].sum()
    
    # How many earliest tokens did we drop?
    n_dropped = m['pivot_token_start'] - 250
    print(f"  Record {idx} ({m['prompt_id']}#{m['sample_idx']})")
    print(f"    pre-pivot tokens: {m['pivot_token_start']}, dropped earliest: {n_dropped}")
    print(f"    Row sum: {row_sum:.4f}")
    print()

Row-sum check across records with different padding levels:

  Record 50 (math_29#9)
    n_real_keys: 249  valid_key_start: 1
    Row sum over real keys: 1.0001

  Record 23 (math_10#1)
    n_real_keys: 161  valid_key_start: 89
    Row sum over real keys: 1.0000

  Record 6 (mis_06#14)
    n_real_keys: 35  valid_key_start: 215
    Row sum over real keys: 1.0001

Records WITHOUT padding (truncated):

  Record 0 (mis_03#8)
    pre-pivot tokens: 282, dropped earliest: 32
    Row sum: 0.8767

  Record 26 (math_12#1)
    pre-pivot tokens: 355, dropped earliest: 105
    Row sum: 0.9063

  Record 28 (math_14#10)
    pre-pivot tokens: 302, dropped earliest: 52
    Row sum: 0.7944



Exactly as expected. The padded records show row sums of 1.0000 — full attention distribution captured. The truncated ones show 0.79–0.91 depending on how much of the early context was dropped.
This isn't a bug. It's the Approach 3 tradeoff working as designed: 19 records truncated, 63 records perfect.

But Phase 4 needs to handle this thoughtfully. We have two choices:

Option A — Normalize weights per row in Phase 4
Divide each row by its sum before analysis. This rescales truncated records so they're comparable to padded ones. Simple, defensible, but loses the magnitude information (e.g., "this query position attends weakly to everything" vs "attends strongly to one thing").

Option B — Use truncation as a control variable
Add truncation_fraction (= 1 - row_sum) as a feature in the classifier. Lets the model learn whether truncation is correlated with the label. This is the more rigorous approach.

I'd recommend Option B because it's the cleaner science. The Phase 4 design we'll write will use it automatically.
Phase 3 v2 is done. Final dataset:

82 records labeled (14 corrections + 68 hallucination_pivots)
Full attention captured for 63 records, partial for 19
~550 MB on disk

Download phase3_attention_v2.npz and phase3_metadata_v2.jsonl from the Kaggle Output tab, then we'll start Phase 4 — the classifier and the four hypothesis tests.

## Run 2 — Attention extraction on the expanded sample (n=134)

This repeats the exact extraction procedure from Run 1 on the combined dataset (batch 1 + batch 2, 134 usable records), producing phase3_attention_v3.npz. Both this output and Run 1's phase3_attention_v2.npz are used in Phase 4 — the paper's central finding is the comparison between them, so both notebooks' outputs must be preserved, not just the larger one.

In [6]:
PHASE2_PATH = '/kaggle/input/notebooks/gunjalgupta/phase-2/pivot_research/phase2_labeled_combined.jsonl'

OUTPUT_DIR = '/kaggle/working/pivot_research'
os.makedirs(OUTPUT_DIR, exist_ok=True)
PHASE3_NPZ_PATH      = f'{OUTPUT_DIR}/phase3_attention_v3.npz'
PHASE3_METADATA_PATH = f'{OUTPUT_DIR}/phase3_metadata_v3.jsonl'

print(f'Input:        {PHASE2_PATH}')
print(f'Output NPZ:   {PHASE3_NPZ_PATH}')
print(f'Output meta:  {PHASE3_METADATA_PATH}')

Input:        /kaggle/input/notebooks/gunjalgupta/phase-2/pivot_research/phase2_labeled_combined.jsonl
Output NPZ:   /kaggle/working/pivot_research/phase3_attention_v3.npz
Output meta:  /kaggle/working/pivot_research/phase3_metadata_v3.jsonl


In [10]:
import json
from collections import Counter

with open(PHASE2_PATH) as f:
    check_records = [json.loads(l) for l in f]

print(f'Total: {len(check_records)}')
for lbl, count in Counter(r['label'] for r in check_records).items():
    print(f'  {lbl:<22} {count}')

Total: 155
  hallucination_pivot    117
  uncertain              15
  correction             23


In [11]:
all_records = []
with open(PHASE2_PATH, 'r') as f:
    for line in f:
        if line.strip():
            all_records.append(json.loads(line))

print(f'Total Phase 2 records: {len(all_records)}')

usable_records = [
    r for r in all_records
    if r['label'] in ('correction', 'hallucination_pivot')
    and r['pivot_token_start'] >= 30
]

from collections import Counter
print(f'Usable records:       {len(usable_records)}')
print()
print('Label distribution:')
for lbl, count in Counter(r['label'] for r in usable_records).items():
    print(f'  {lbl:<22} {count}')

pre_pivot_lens = sorted([r['pivot_token_start'] for r in usable_records])
print(f'\nPre-pivot token counts (= max keys available):')
print(f'  Min:    {pre_pivot_lens[0]}')
print(f'  Median: {pre_pivot_lens[len(pre_pivot_lens)//2]}')
print(f'  Max:    {pre_pivot_lens[-1]}')

n_will_truncate = sum(1 for n in pre_pivot_lens if n > 250)
n_will_pad      = sum(1 for n in pre_pivot_lens if n < 250)
print(f'\nWith MAX_KEY_CONTEXT = 250:')
print(f'  Records truncated (>250 pre-pivot): {n_will_truncate}')
print(f'  Records padded (<250 pre-pivot):    {n_will_pad}')
print(f'  Records full-coverage (=250):       {len(usable_records) - n_will_truncate - n_will_pad}')

Total Phase 2 records: 155
Usable records:       134

Label distribution:
  hallucination_pivot    112
  correction             22

Pre-pivot token counts (= max keys available):
  Min:    34
  Median: 149
  Max:    380

With MAX_KEY_CONTEXT = 250:
  Records truncated (>250 pre-pivot): 23
  Records padded (<250 pre-pivot):    111
  Records full-coverage (=250):       0


In [13]:
!pip install -q transformer_lens transformers torch

import torch
import numpy as np
import json
import os
from pathlib import Path

assert torch.cuda.is_available(), 'No GPU detected. Session options -> Accelerator -> GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 87.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 36.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.6

In [14]:
from transformer_lens import HookedTransformer

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f'Loading {MODEL_NAME} via TransformerLens...')
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    device='cuda',
    dtype=torch.float16,
)
model.eval()

N_LAYERS = model.cfg.n_layers
N_HEADS  = model.cfg.n_heads
D_HEAD   = model.cfg.d_head
D_MODEL  = model.cfg.d_model

print(f'\nLayers: {N_LAYERS}, Heads: {N_HEADS}, D_head: {D_HEAD}')

/tmp/ipykernel_58/2086704808.py:6: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained(


Loading Qwen/Qwen2.5-1.5B-Instruct via TransformerLens...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-1.5B-Instruct into HookedTransformer

Layers: 28, Heads: 12, D_head: 128


In [15]:
QUERY_WINDOW_SIZE = 30
MAX_KEY_CONTEXT   = 250

def extract_attention_features_v2(record: dict) -> dict:
    text   = record['generated_text']
    tokens = model.tokenizer.encode(text, add_special_tokens=False, return_tensors='pt').to('cuda')
    seq_len = tokens.shape[1]

    pivot_idx = record['pivot_token_start']
    if pivot_idx >= seq_len:
        raise ValueError(f'Pivot {pivot_idx} >= seq_len {seq_len}')

    query_start = pivot_idx - QUERY_WINDOW_SIZE
    query_end   = pivot_idx

    key_real_start = max(0, pivot_idx - MAX_KEY_CONTEXT)
    key_end        = pivot_idx
    n_real_keys    = key_end - key_real_start
    valid_key_start = MAX_KEY_CONTEXT - n_real_keys

    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)

    attn_weights_per_layer = []
    for layer in range(N_LAYERS):
        pattern = cache['pattern', layer][0]
        sliced = pattern[:, query_start:query_end, key_real_start:key_end]
        padded = torch.zeros(
            (N_HEADS, QUERY_WINDOW_SIZE, MAX_KEY_CONTEXT),
            dtype=sliced.dtype, device=sliced.device
        )
        padded[:, :, valid_key_start:] = sliced
        attn_weights_per_layer.append(padded.cpu().float().numpy())

    attn_weights = np.stack(attn_weights_per_layer, axis=0)

    attn_outputs_per_layer = []
    for layer in range(N_LAYERS):
        z = cache['z', layer][0]
        window_z = z[query_start:query_end]
        attn_outputs_per_layer.append(window_z.cpu().float().numpy())

    attn_outputs = np.stack(attn_outputs_per_layer, axis=0)

    del cache
    torch.cuda.empty_cache()

    return {
        'attn_weights': attn_weights, 'attn_outputs': attn_outputs,
        'pivot_idx': pivot_idx, 'seq_len': seq_len,
        'valid_key_start': valid_key_start, 'n_real_keys': n_real_keys,
    }

import time
print('Testing extraction on first record...')
t0 = time.time()
test_result = extract_attention_features_v2(usable_records[0])
elapsed = time.time() - t0
print(f'Took {elapsed:.2f}s')
print(f"weights shape: {test_result['attn_weights'].shape}")
print(f"outputs shape: {test_result['attn_outputs'].shape}")
print(f"Estimated total time: {elapsed * len(usable_records) / 60:.1f} minutes")

Testing extraction on first record...
Took 0.83s
weights shape: (28, 12, 30, 250)
outputs shape: (28, 30, 12, 128)
Estimated total time: 1.9 minutes


In [16]:
from tqdm.auto import tqdm

all_weights, all_outputs, all_metadata, errors = {}, {}, [], []
start_time = time.time()

for i, record in enumerate(tqdm(usable_records, desc='Extracting attention v3')):
    try:
        result = extract_attention_features_v2(record)
        all_weights[f'weights_{i:03d}'] = result['attn_weights']
        all_outputs[f'outputs_{i:03d}'] = result['attn_outputs']
        all_metadata.append({
            'idx': i, 'prompt_id': record['prompt_id'], 'category': record['category'],
            'sample_idx': record['sample_idx'], 'label': record['label'],
            'pivot_type': record['pivot_type'], 'pivot_phrase': record['pivot_phrase'],
            'pivot_token_start': record['pivot_token_start'],
            'query_window_size': QUERY_WINDOW_SIZE, 'max_key_context': MAX_KEY_CONTEXT,
            'valid_key_start': result['valid_key_start'], 'n_real_keys': result['n_real_keys'],
            'seq_len': result['seq_len'],
        })
    except Exception as e:
        print(f'\nERROR on record {i}: {e}')
        errors.append({'idx': i, 'error': str(e)})

elapsed = time.time() - start_time
print(f'\nDone in {elapsed/60:.1f} min. Successful: {len(all_metadata)}, Errors: {len(errors)}')

Extracting attention v3:   0%|          | 0/134 [00:00<?, ?it/s]


Done in 0.5 min. Successful: 134, Errors: 0


In [17]:
PHASE3_NPZ_PATH      = '/kaggle/working/pivot_research/phase3_attention_v3.npz'
PHASE3_METADATA_PATH = '/kaggle/working/pivot_research/phase3_metadata_v3.jsonl'

np.savez_compressed(PHASE3_NPZ_PATH, **all_weights, **all_outputs)
with open(PHASE3_METADATA_PATH, 'w') as f:
    for m in all_metadata:
        f.write(json.dumps(m) + '\n')

weights_size = os.path.getsize(PHASE3_NPZ_PATH) / 1e6
meta_size    = os.path.getsize(PHASE3_METADATA_PATH) / 1e6
print(f'{PHASE3_NPZ_PATH} ({weights_size:.1f} MB)')
print(f'{PHASE3_METADATA_PATH} ({meta_size:.2f} MB)')

from collections import Counter
counts = Counter(m['label'] for m in all_metadata)
for lbl, count in counts.items():
    print(f'  {lbl:<22} {count}')

/kaggle/working/pivot_research/phase3_attention_v3.npz (856.6 MB)
/kaggle/working/pivot_research/phase3_metadata_v3.jsonl (0.04 MB)
  hallucination_pivot    112
  correction             22


In [18]:
loaded = np.load(PHASE3_NPZ_PATH)
print(f'Loaded {len(loaded.keys())} arrays')

w0   = loaded['weights_000']
meta = all_metadata[0]
vks  = meta['valid_key_start']

print(f'\nRecord 0: {meta["prompt_id"]}#{meta["sample_idx"]}')
print(f'  weights shape:  {w0.shape}')
print(f'  valid_key_start: {vks}')

query_pos = 15
real_keys = w0[0, 0, query_pos, vks:]
print(f'\nRow sum over real keys: {real_keys.sum():.4f}')
print(f'Padding region sum: {w0[0, 0, query_pos, :vks].sum():.4f}')

Loaded 268 arrays

Record 0: mis_03#8
  weights shape:  (28, 12, 30, 250)
  valid_key_start: 0

Row sum over real keys: 0.8767
Padding region sum: 0.0000
